# EV3 — Análisis Exploratorio de Datos (EDA)
### Sistema de Recomendación de Hardware para PC

Este notebook cruza tres fuentes de datos para generar recomendaciones de hardware:
- **CSV (ETL):** Requisitos de juegos y popularidad de hardware en Steam
- **MySQL (BD):** Catálogo de componentes, tiers y builds pre-armadas
- **API eBay:** Precios reales de componentes en CLP

## 1. Importaciones y Conexiones

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import mysql.connector
import os
import warnings
warnings.filterwarnings('ignore')

# Rutas relativas desde la carpeta /eda
BASE = os.path.join(os.path.dirname(os.getcwd()), '') if os.path.basename(os.getcwd()) == 'eda' else ''
KAGGLE_CSV      = os.path.join(BASE, 'data', 'kaggle', 'games_sample_15.csv')
STEAM_CSV       = os.path.join(BASE, 'data', 'steamhwsurvey', 'steam_sample_15.csv')
BUILDS_CSV      = os.path.join(BASE, 'data', 'builds_populares.csv')
GAMES_PRICED    = os.path.join(BASE, 'data', 'kaggle', 'games_sample_15_PRICED.csv')
STEAM_PRICED    = os.path.join(BASE, 'data', 'steamhwsurvey', 'steam_sample_15_PRICED.csv')

print('Rutas configuradas correctamente.')

In [ ]:
# Conexión a MySQL
try:
    conn = mysql.connector.connect(
        host='localhost',
        port=3306,
        user='root',
        password='',
        database='tienda_hardware_intelligence'
    )
    print('Conexión a MySQL exitosa.')
except Exception as e:
    print(f'Error de conexión: {e}')
    conn = None

## 2. Carga de Datos

In [ ]:
# --- FUENTE 1: CSV (ETL) ---
df_games  = pd.read_csv(KAGGLE_CSV)
df_steam  = pd.read_csv(STEAM_CSV)
df_builds = pd.read_csv(BUILDS_CSV)

# CSV con precios inyectados por la API de eBay (generados por fetch_prices.py)
df_games_priced = pd.read_csv(GAMES_PRICED) if os.path.exists(GAMES_PRICED) else df_games.copy()
df_steam_priced = pd.read_csv(STEAM_PRICED) if os.path.exists(STEAM_PRICED) else df_steam.copy()

print(f'Juegos cargados: {len(df_games)}')
print(f'Datos Steam cargados: {len(df_steam)}')
print(f'Builds cargadas: {len(df_builds)}')

In [ ]:
# --- FUENTE 2: MySQL (BD) ---
if conn:
    df_components = pd.read_sql('SELECT c.id, c.name, c.categoria, t.tier_name FROM component c JOIN component_tiers t ON c.component_tiers_id = t.id', conn)
    df_prices_db  = pd.read_sql('SELECT c.name as componente, m.price_clp FROM market_prices_external m JOIN component c ON m.component_id = c.id', conn)
    df_builds_db  = pd.read_sql('SELECT bt.template_name, c.name as componente, c.categoria, t.tier_name FROM build_templates bt JOIN build_components bc ON bt.id = bc.build_templates_id JOIN component c ON bc.component_id = c.id JOIN component_tiers t ON c.component_tiers_id = t.id', conn)
    df_steam_db   = pd.read_sql('SELECT c.name as componente, s.global_share_percentage as porcentaje FROM steam_hardware_survey s JOIN component c ON s.component_id = c.id', conn)
    df_games_db   = pd.read_sql('SELECT g.titulo as juego, c.name as componente, c.categoria, gr.requirement_type FROM games g JOIN game_requeriments gr ON g.id = gr.games_id JOIN component c ON gr.component_id = c.id', conn)
    print('Tablas MySQL cargadas correctamente.')
else:
    print('Sin conexión MySQL. Usando solo CSVs.')

## 3. Vista Previa de los Datos

In [ ]:
print('=== JUEGOS (CSV) ===')
display(df_games.head())
print('\n=== STEAM HW SURVEY (CSV) ===')
display(df_steam.head())
print('\n=== COMPONENTES (MySQL) ===')
display(df_components)

---
## 4. Análisis 1 — ¿Qué exige el mercado de juegos?
Analizamos los requisitos de los 15 juegos más populares agrupados por calidad objetivo.

In [ ]:
# Conteo de juegos por target_performance
conteo_target = df_games['target_performance'].value_counts().reset_index()
conteo_target.columns = ['Calidad Objetivo', 'Cantidad de Juegos']

fig = px.bar(
    conteo_target,
    x='Calidad Objetivo', y='Cantidad de Juegos',
    color='Calidad Objetivo',
    title='Distribución de Juegos por Calidad Objetivo',
    template='plotly_white',
    text='Cantidad de Juegos'
)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# GPUs más solicitadas por los juegos (CSV)
gpus_juegos = df_games['gpu'].str.split(' o ').explode().str.strip()
top_gpus_juegos = gpus_juegos.value_counts().head(10).reset_index()
top_gpus_juegos.columns = ['GPU', 'Juegos que la piden']

fig2 = px.bar(
    top_gpus_juegos,
    x='Juegos que la piden', y='GPU',
    orientation='h',
    title='Top GPUs más exigidas por los juegos (Requisitos Recomendados)',
    template='plotly_white',
    color='Juegos que la piden',
    color_continuous_scale='Blues'
)
fig2.show()

In [ ]:
# RAM más solicitada
ram_dist = df_games['ram'].value_counts().reset_index()
ram_dist.columns = ['RAM Requerida', 'Cantidad']

fig3 = px.pie(
    ram_dist,
    names='RAM Requerida', values='Cantidad',
    title='Distribución de RAM requerida en los 15 juegos',
    hole=0.4,
    template='plotly_white'
)
fig3.show()

---
## 5. Análisis 2 — ¿Qué hardware usa el mundo? (Steam HW Survey)

In [ ]:
# Hardware más popular según Steam (desde MySQL)
if conn and 'df_steam_db' in dir():
    fig4 = px.bar(
        df_steam_db.sort_values('porcentaje', ascending=True),
        x='porcentaje', y='componente',
        orientation='h',
        title='Popularidad de Hardware en Steam (% usuarios globales)',
        template='plotly_white',
        color='porcentaje',
        color_continuous_scale='Greens',
        labels={'porcentaje': '% de usuarios', 'componente': 'Componente'}
    )
    fig4.show()
else:
    # Fallback con CSV
    fig4 = px.bar(
        df_steam.sort_values('percentage', ascending=True),
        x='percentage', y='name',
        orientation='h',
        title='Popularidad de Hardware en Steam (% usuarios globales)',
        template='plotly_white',
        color='percentage',
        color_continuous_scale='Greens'
    )
    fig4.show()

---
## 6. Análisis 3 — Precios de componentes (API eBay)
¿Cuánto cuesta actualizar cada componente hoy en el mercado?

In [ ]:
# Precios desde MySQL (eBay via BD)
if conn and 'df_prices_db' in dir() and len(df_prices_db) > 0:
    df_prices_db['price_clp_miles'] = (df_prices_db['price_clp'] / 1000).round(0)
    fig5 = px.bar(
        df_prices_db.sort_values('price_clp', ascending=False),
        x='componente', y='price_clp',
        title='Precio de Componentes en el Mercado (CLP) — Fuente: eBay API',
        template='plotly_white',
        color='price_clp',
        color_continuous_scale='Reds',
        labels={'price_clp': 'Precio (CLP)', 'componente': 'Componente'},
        text=df_prices_db.sort_values('price_clp', ascending=False)['price_clp'].apply(lambda x: f'${x:,.0f}')
    )
    fig5.update_traces(textposition='outside')
    fig5.update_xaxes(tickangle=30)
    fig5.show()

# Precios desde CSV PRICED (eBay via API directa)
if os.path.exists(GAMES_PRICED):
    dolar = 950  # CLP por USD aproximado
    cols_precio = ['gpu', 'price_gpu_usd']
    if all(c in df_games_priced.columns for c in cols_precio):
        df_gpu_prices = df_games_priced[['game_name','gpu','price_gpu_usd']].dropna()
        df_gpu_prices['price_gpu_clp'] = (df_gpu_prices['price_gpu_usd'] * dolar).round(0)
        fig5b = px.bar(
            df_gpu_prices.sort_values('price_gpu_clp', ascending=False),
            x='game_name', y='price_gpu_clp',
            title='Precio GPU requerida por cada juego (CLP) — Fuente: eBay API',
            template='plotly_white',
            color='price_gpu_clp',
            color_continuous_scale='Oranges',
            labels={'price_gpu_clp': 'Precio GPU (CLP)', 'game_name': 'Juego'}
        )
        fig5b.update_xaxes(tickangle=45)
        fig5b.show()

---
## 7. Análisis 4 — Cruce Principal: ¿Cuánto cuesta armar cada Build?
Comparamos el costo total de cada build pre-armada según los precios de eBay.

In [ ]:
# Costo de cada build desde MySQL
if conn and 'df_builds_db' in dir() and 'df_prices_db' in dir():
    df_build_cost = df_builds_db.merge(df_prices_db, left_on='componente', right_on='componente', how='left')
    df_build_total = df_build_cost.groupby('template_name')['price_clp'].sum().reset_index()
    df_build_total.columns = ['Build', 'Costo Total (CLP)']
    df_build_total = df_build_total.sort_values('Costo Total (CLP)', ascending=False)

    fig6 = px.bar(
        df_build_total,
        x='Build', y='Costo Total (CLP)',
        title='Costo Total de Armado por Build (CLP) — Precios eBay',
        template='plotly_white',
        color='Build',
        text=df_build_total['Costo Total (CLP)'].apply(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')
    )
    fig6.update_traces(textposition='outside')
    fig6.show()

    display(df_build_cost[['template_name','componente','categoria','tier_name','price_clp']])

---
## 8. Análisis 5 — ¿Qué juegos puede correr el PC Promedio de Steam?

In [ ]:
# Componentes del "PC Promedio de Steam" (Build 1 de nuestro CSV)
pc_promedio = df_builds[df_builds['build_name'] == 'El PC Promedio de Steam'].iloc[0]
print('PC Promedio de Steam:')
print(pc_promedio.to_string())

# Clasificamos cada juego como "puede correr" o "necesita upgrade"
# (comparación simplificada basada en RAM y target_performance)
def puede_correr(row):
    ram_juego = int(str(row['ram']).replace('GB','').strip().split()[0])
    ram_pc = int(str(pc_promedio['ram']).replace('GB','').strip().split()[0])
    target = row['target_performance']
    if ram_juego > ram_pc:
        return 'Necesita más RAM'
    elif '144fps' in target:
        return '✅ Puede correr (eSports)'
    else:
        return '⚠️ Puede correr (ajustes medios)'

df_games['estado_pc_promedio'] = df_games.apply(puede_correr, axis=1)

fig7 = px.bar(
    df_games,
    x='game_name', y=[1]*len(df_games),
    color='estado_pc_promedio',
    title='¿Puede el PC Promedio de Steam correr estos juegos?',
    template='plotly_white',
    labels={'y': 'Juego', 'game_name': 'Juego', 'color': 'Estado'},
    color_discrete_map={
        '✅ Puede correr (eSports)': '#2ecc71',
        '⚠️ Puede correr (ajustes medios)': '#f39c12',
        'Necesita más RAM': '#e74c3c'
    }
)
fig7.update_xaxes(tickangle=45)
fig7.update_yaxes(showticklabels=False, title='')
fig7.show()

print('\nResumen:')
print(df_games[['game_name','ram','target_performance','estado_pc_promedio']])

---
## 9. Análisis 6 — Componentes por Gama (Tiers)
Distribución del catálogo de componentes por nivel de rendimiento.

In [ ]:
if conn and 'df_components' in dir():
    tier_dist = df_components.groupby(['categoria','tier_name']).size().reset_index(name='cantidad')
    fig8 = px.bar(
        tier_dist,
        x='categoria', y='cantidad',
        color='tier_name',
        barmode='group',
        title='Distribución de Componentes por Categoría y Gama',
        template='plotly_white',
        labels={'categoria': 'Tipo de Componente', 'cantidad': 'Cantidad', 'tier_name': 'Gama'},
        color_discrete_map={'Gama Baja': '#3498db', 'Gama Media': '#2ecc71', 'Gama Alta': '#e74c3c'}
    )
    fig8.show()
    display(df_components)

---
## 10. Conclusiones

### Hallazgos principales:

1. **Demanda del mercado:** La mayoría de los juegos AAA requieren una GPU de Gama Media (RTX 3060 / RX 6600) para correr a 1080p/60fps en calidad alta.

2. **El PC promedio global (Steam):** La RTX 3060 es la GPU más popular, y el 42% de usuarios ya tiene 16GB de RAM — lo que indica que el mercado está migrando hacia ese estándar.

3. **Brecha de actualización:** Los juegos eSports (Valorant, CS:GO, LoL) corren perfectamente en hardware de gama baja/media, pero los AAA (Cyberpunk, Hogwarts Legacy) requieren una inversión real.

4. **Precios (eBay):** La GPU es el componente más caro y tiene mayor impacto en la capacidad de juego. Representa entre el 40-60% del costo total de un build.

5. **Build recomendada:** El "PC Gamer Ultra 1080p" ofrece la mejor relación costo/beneficio para cubrir la mayoría de los juegos de la muestra.

In [ ]:
# Cerrar conexión MySQL
if conn:
    conn.close()
    print('Conexión MySQL cerrada.')